# Milestone 2 slice 2 — Encoder frame classification (DeBERTa-v3-large)

Fine-tune `microsoft/deberta-v3-large` to classify which FrameNet frame a
trigger evokes. The trigger word is wrapped in entity markers
(`The chef <t> gave </t> food.`); at inference the logits are **masked to the
lexicon's candidate frames**, so an invalid frame can't be produced.

**Baseline to beat (frame classification):** F1 `0.887` (our reproduction) /
`0.89` reported.

Same robustness rules as the trigger notebook (clone don't pip-build; leave
torch/protobuf/numpy alone; protobuf env var before imports). Use a **fresh
GPU runtime**.

> `Runtime → GPU` (A100), then `Run all`.

In [ ]:
!nvidia-smi

In [ ]:
# --- Clone the private project repo + install pinned env --------------------
import os
try:
    from google.colab import userdata
    os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN") or ""
except Exception:
    os.environ.setdefault("GH_TOKEN", "")  # or set manually

![ -d Texture_Frames ] && (cd Texture_Frames && git pull -q) || git clone -q https://$GH_TOKEN@github.com/texturejc/Texture_Frames.git
!cd Texture_Frames && echo "project at $(git rev-parse --short HEAD)"
!pip install -q -r Texture_Frames/requirements-colab.txt

In [ ]:
# Version sanity check — numpy MUST be 2.x. If 1.x: DELETE runtime (not Restart).
import numpy, sys
print("numpy      :", numpy.__version__)
import numpy.strings  # raises on numpy 1.x
import torch, transformers
print("torch      :", torch.__version__, "| transformers:", transformers.__version__)
print("cuda       :", torch.cuda.is_available())

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"  # before transformers
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")

import nltk
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## Sanity check — lexicon candidates + marked input

Building the lexicon (stemming ~1221 frames' lexical units) takes a minute the
first time. Confirm the candidate frames and entity markers look right before
the multi-hour train.

In [ ]:
import data
from lexicon import Lexicon

lex = Lexicon()
print("frame vocabulary size:", len(lex.frame2id()))

examples = data.load_frame_examples("dev")
print("dev frame examples:", len(examples))
text, loc, gold = examples[0]
print("\ntext        :", text)
print("marked      :", data.mark_trigger(text, loc))
print("gold frame  :", gold)
cands = lex.candidate_frames(text, loc)
print("candidates  :", cands)
print("gold in candidates?", gold in cands)

## Train

In [ ]:
import train_frame

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/texture_frames/frame"
except Exception:
    OUTPUT_DIR = "outputs/frame"
print("saving to:", OUTPUT_DIR)

model, tokenizer, lexicon = train_frame.train(
    base_model="microsoft/deberta-v3-large",
    output_dir=OUTPUT_DIR,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=320,
)

## Evaluate — candidate-masked frame accuracy (baseline-comparable)

In [ ]:
from eval_frame import evaluate_frame, print_report

metrics = evaluate_frame(model, tokenizer, lexicon, split="test", max_length=320)
print_report(metrics, reported_f1=0.887)

## Interpreting

Report **frame F1/accuracy vs 0.887** and the **lexicon coverage** number.
Coverage = share of gold frames that appear in the candidate set; it's the
ceiling on accuracy (a gold frame never in the candidates can't be predicted
once masking is on). If coverage is high (~0.95+) and accuracy is near it, the
head is doing its job on the ambiguous multi-candidate triggers.